# Perplexity and Linguistic Novelty

This notebook focuses on testing if perplexity scores (Zhang and Evans, 2025) can Be manipulated through linguistic novelty. We try to keep the conceptual content constant and chang the linguistic novelty by three components: Jargon Manipulation, Syntax Manipulation, and Complexity Manipulation. 



In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import warnings
warnings.filterwarnings('ignore')

We are using the same dataset, `macss_abstracts_perplexity_cleaned.csv` we created in `Week1_and_2_exploration.ipynb`

In [59]:
df = pd.read_csv("data/macss_abstracts_perplexity_cleaned.csv")
df

,paper_id,title,abstract,year,Author,clean_abstract,perplexity_score,num_tokens
0,macss_01,News to Numbers: A Comparative Analysis of Tra...,This thesis investigates the comparative effec...,2025,"Bai, Rong",this thesis investigates the comparative effec...,59.347969,558
1,macss_02,Labor Market Shocks and Heterogeneous Adjustme...,This study examines how labor markets adjust t...,2025,"Xu, Jingzhi",this study examines how labor markets adjust t...,74.344353,152
2,macss_03,The Cost of Decentralization? Welfare Comparis...,This paper quantifies the welfare implications...,2025,"Bhatt, Nalin",this paper quantifies the welfare implications...,70.330452,290
3,macss_04,The Art of Positivity in Drawing: Unveiling th...,This study integrates visual creative tasks wi...,2025,"Cong, Tianyue",this study integrates visual creative tasks wi...,66.121506,411
4,macss_05,Mapping Informality and Violence: Machine Lear...,This study examines the relationship between n...,2025,"Liang, Yingyi",this study examines the relationship between n...,43.541126,180
...,...,...,...,...,...,...,...,...
258,macss_266,Refugee Inflow and Native Female Labor Supply,This project investigates the impacts of refug...,2024,"Liu, Sifei",this project investigates the impacts of refug...,33.083405,200
259,macss_267,"Swipe Left, or Right? Decoding Generative AI'...",Dating apps shape romantic opportunities but r...,2025,"Li, Aidi",dating apps shape romantic opportunities but r...,122.738571,188
260,macss_268,A Tale of Two Monetary Transmissions: Evidence...,Monetary authorities use contractionary moneta...,2023,"Fang, Chongyu",monetary authorities use contractionary moneta...,40.773228,212
261,macss_269,What people worry about: Top Personal Finance ...,Personal finance problems have become a signif...,2022,"Jinfei, Zhu",personal finance problems have become a signif...,111.945839,111


In [60]:
# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Load perplexity calculation model
print("Loading GPT-2 for perplexity calculation...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
model.eval()
print(f"Using device: {device}")


Loading GPT-2 for perplexity calculation...
Using device: cpu


**Linguistic Jargon Manipulation**

In [61]:
def linguistic_manipulation_jargon(text, seed=None):
      """Add excessive jargon to increase linguistic complexity."""
      if seed is not None:
          np.random.seed(seed)

      jargon_phrases = [
          "leveraging synergistic paradigms",
          "optimizing cross-functional deliverables",
          "via heterogeneous methodological frameworks",
          "employing multidimensional analytical constructs",
          "through iterative epistemic procedures"
      ]
      # Insert jargon phrases at sentence boundaries
      # these can be modified!! 
      sentences = text.split('. ')
      manipulated = []
      for i, sent in enumerate(sentences):
          manipulated.append(sent)
          if i < len(sentences) - 1 and i % 2 == 0:
              manipulated.append(np.random.choice(jargon_phrases))
      return '. '.join(manipulated)

**Linguistic Syntax Mainpulation**

In [62]:
def linguistic_manipulation_syntax(text, seed=None):
      """Create unusual syntax patterns while preserving meaning."""
      if seed is not None:
          np.random.seed(seed)

      sentences = text.split('. ')
      manipulated = []
      for sent in sentences:
          words = sent.split()
          if len(words) > 5:
            # Random reordering while maintaining some grammatical structure
            # Move adjectives/adverbs to unusual positions
              if np.random.random() > 0.5:
                  # Invert subject-verb in some cases (mimicking non-English syntax)
                  mid = len(words) // 2
                  sent = ' '.join(words[mid:] + words[:mid])
          manipulated.append(sent)
      return '. '.join(manipulated)

**Linguistic Complexity Manipulation**

In [63]:
def linguistic_manipulation_complexity(text, seed=None):
      """Increase sentence complexity through nested clauses."""
      if seed is not None:
          np.random.seed(seed)

      sentences = text.split('. ')
      complex_connectors = [
          "notwithstanding the aforementioned considerations",
          "wherein the preponderance of evidence suggests",
          "insofar as the empirical findings demonstrate",
          "whereby the theoretical framework posits"
      ]
      manipulated = []
      for i, sent in enumerate(sentences):
          if i > 0 and i % 2 == 1:
              sent = f"{np.random.choice(complex_connectors)}, {sent.lower()}"
          manipulated.append(sent)
      return '. '.join(manipulated)

### Linguistic Manipulation

In [64]:
df['perplexity_category'] = pd.cut(df['perplexity_score'],
                                    bins=[0, 50, 70, 100, 200],
                                    labels=['Low (0-50)', 'Medium (50-70)', 'High (70-100)', 'Very High (100+)'])
print(df['perplexity_category'].value_counts())

perplexity_category
Low (0-50)          119
Medium (50-70)       79
High (70-100)        43
Very High (100+)     20
Name: count, dtype: int64


We will first explore 5 samples from each category 

In [65]:
sample_df = df.groupby('perplexity_category', group_keys=False).apply(
    lambda x: x.sample(min(5, len(x)), random_state=42)
)


In [66]:
from calculate_perplexity import calculate_perplexity_and_tokens

def compute_perplexity(text, uid="notebook_row", max_length=2048, return_tokens=False):
    resolved_device = device if "device" in globals() else next(model.parameters()).device
    ppl, n_tokens = calculate_perplexity_and_tokens(
        text=text,
        uid=uid,
        model=model,
        tokenizer=tokenizer,
        device=resolved_device,
        max_length=max_length,
    )
    return (ppl, n_tokens) if return_tokens else ppl


In [67]:
results = []

for idx, row in sample_df.iterrows():
    base_text = row['abstract'] if ('abstract' in row and pd.notna(row['abstract'])) else row['clean_abstract']
    uid = str(row['paper_id']) if 'paper_id' in row else str(idx)
    original_ppl = compute_perplexity(base_text, uid=f"orig_{uid}")

    print(f"\nProcessing: {row['title'][:50]}...")
    print(f"Original perplexity (recomputed): {original_ppl:.2f}")

    # Use row index as seed for reproducibility
    seed = idx

    # Original
    results.append({
        'paper_id': row['paper_id'],
        'title': row['title'],
        'manipulation': 'Original',
        'perplexity': original_ppl,
        'manipulated_text': base_text,
        'original_text': base_text
    })

    # Manipulation 1: Jargon
    manip_text_jargon = linguistic_manipulation_jargon(base_text, seed=seed)
    ppl_jargon = compute_perplexity(manip_text_jargon, uid=f"jargon_{uid}")
    results.append({
        'paper_id': row['paper_id'],
        'title': row['title'],
        'manipulation': 'Jargon',
        'perplexity': ppl_jargon,
        'manipulated_text': manip_text_jargon,
        'original_text': base_text
    })
    print(f"  Jargon: {ppl_jargon:.2f} ({ppl_jargon - original_ppl:+.2f})")

    # Manipulation 2: Syntax
    manip_text_syntax = linguistic_manipulation_syntax(base_text, seed=seed)
    ppl_syntax = compute_perplexity(manip_text_syntax, uid=f"syntax_{uid}")
    results.append({
        'paper_id': row['paper_id'],
        'title': row['title'],
        'manipulation': 'Syntax',
        'perplexity': ppl_syntax,
        'manipulated_text': manip_text_syntax,
        'original_text': base_text
    })
    print(f"  Syntax: {ppl_syntax:.2f} ({ppl_syntax - original_ppl:+.2f})")

    # Manipulation 3: Complexity
    manip_text_complexity = linguistic_manipulation_complexity(base_text, seed=seed)
    ppl_complexity = compute_perplexity(manip_text_complexity, uid=f"complexity_{uid}")
    results.append({
        'paper_id': row['paper_id'],
        'title': row['title'],
        'manipulation': 'Complexity',
        'perplexity': ppl_complexity,
        'manipulated_text': manip_text_complexity,
        'original_text': base_text
    })
    print(f"  Complexity: {ppl_complexity:.2f} ({ppl_complexity - original_ppl:+.2f})")



Processing: Uncovering Market Trends Through Topic Modeling an...
Original perplexity (recomputed): 30.11
  Jargon: 31.20 (+1.10)
  Syntax: 45.15 (+15.04)
  Complexity: 43.13 (+13.03)

Processing: Social Media, Misogyny, and Labor Market Outcomes...
Original perplexity (recomputed): 38.25
  Jargon: 44.23 (+5.98)
  Syntax: 50.21 (+11.97)
  Complexity: 51.02 (+12.77)

Processing: Emotional and Cultural Patterns Expressed in Chine...
Original perplexity (recomputed): 34.95
  Jargon: 43.30 (+8.35)
  Syntax: 54.52 (+19.57)
  Complexity: 41.96 (+7.01)

Processing: Embracing New Techniques in Deep learning for Pred...
Original perplexity (recomputed): 27.10
  Jargon: 36.78 (+9.68)
  Syntax: 44.65 (+17.55)
  Complexity: 33.59 (+6.49)

Processing: Diagnostic images for mild cognitive impairment re...
Original perplexity (recomputed): 24.42
  Jargon: 27.35 (+2.93)
  Syntax: 37.28 (+12.85)
  Complexity: 33.73 (+9.30)

Processing: Assessing the Predictive Power of Urban Green Spac...
Original per

In [68]:
results_df = pd.DataFrame(results)
results_df.to_csv('data/perplexity_manipulation_results_modified.csv', index=False)

**Statistical Summary**

In [69]:
summary = results_df.groupby('manipulation')['perplexity'].agg(['mean', 'std', 'min', 'max', 'count'])
print(summary)

# Calculate percentage change from original
original_mean = results_df[results_df['manipulation'] == 'Original']['perplexity'].mean()
for manip in ['Jargon', 'Syntax', 'Complexity']:
    manip_mean = results_df[results_df['manipulation'] == manip]['perplexity'].mean()
    pct_change = ((manip_mean - original_mean) / original_mean) * 100
    print(f"\n{manip} manipulation: {pct_change:+.2f}% change in perplexity")

                   mean        std        min         max  count
manipulation                                                    
Complexity    65.199681  22.316711  33.586620  114.464958     20
Jargon        59.738755  21.501064  27.350777  106.843567     20
Original      53.803610  19.867146  24.424702   96.000641     20
Syntax        76.495640  31.731857  37.278687  149.737457     20

Jargon manipulation: +11.03% change in perplexity

Syntax manipulation: +42.18% change in perplexity

Complexity manipulation: +21.18% change in perplexity


In [70]:
for manip in ['Jargon', 'Syntax', 'Complexity']:
    original_scores = results_df[results_df['manipulation'] == 'Original']['perplexity'].values
    manip_scores = results_df[results_df['manipulation'] == manip]['perplexity'].values
    differences = manip_scores - original_scores
    print(f"\n{manip}:")
    print(f"  Mean difference: {differences.mean():+.2f}")
    print(f"  Std difference: {differences.std():.2f}")
    print(f"  Min/Max difference: {differences.min():+.2f} / {differences.max():+.2f}")


Jargon:
  Mean difference: +5.94
  Std difference: 4.32
  Min/Max difference: -0.83 / +16.14

Syntax:
  Mean difference: +22.69
  Std difference: 15.53
  Min/Max difference: +0.00 / +53.74

Complexity:
  Mean difference: +11.40
  Std difference: 9.03
  Min/Max difference: -4.49 / +34.39


### Case Exploration

In [71]:
# calculate difference from original abstract
results_with_diff = []
for paper_id in results_df['paper_id'].unique():
    paper_data = results_df[results_df['paper_id'] == paper_id]
    original_ppl = paper_data[paper_data['manipulation'] == 'Original']['perplexity'].values[0]
    title = paper_data['title'].values[0]

    for manip in ['Jargon', 'Syntax', 'Complexity']:
        manip_data = paper_data[paper_data['manipulation'] == manip]
        if len(manip_data) > 0:
            manip_ppl = manip_data['perplexity'].values[0]
            diff = manip_ppl - original_ppl
            results_with_diff.append({
                'paper_id': paper_id,
                'title': title,
                'manipulation': manip,
                'original_ppl': original_ppl,
                'manipulated_ppl': manip_ppl,
                'difference': diff,
                'pct_change': (diff / original_ppl) * 100
            })

diff_df = pd.DataFrame(results_with_diff)

**Top Cases**

In [72]:
top_cases = diff_df.nlargest(3, 'difference')
for idx, case in top_cases.iterrows():
    print(f"\n{case['manipulation']} Manipulation:")
    print(f"  Title: {case['title'][:60]}...")
    print(f"  Original PPL: {case['original_ppl']:.2f}")
    print(f"  Manipulated PPL: {case['manipulated_ppl']:.2f}")
    print(f"  Difference: +{case['difference']:.2f} ({case['pct_change']:+.1f}%)")

    # Get the actual text
    paper_results = results_df[results_df['paper_id'] == case['paper_id']]
    original_text = paper_results[paper_results['manipulation'] == 'Original']['original_text'].values[0]
    manip_text = paper_results[paper_results['manipulation'] == case['manipulation']]['manipulated_text'].values[0]

    print(f"\n  ORIGINAL TEXT:")
    print(f"  {original_text}...")
    print(f"\n  MANIPULATED TEXT:")
    print(f"  {manip_text}...")


Syntax Manipulation:
  Title: Differential impact from individual versus collective misinf...
  Original PPL: 96.00
  Manipulated PPL: 149.74
  Difference: +53.74 (+56.0%)

  ORIGINAL TEXT:
  Fears about the destabilizing impact of misinformation online have motivated individuals and platforms to respond. Individuals have increasingly challenged others’ online claims with fact-checks in pursuit of a healthier information ecosystem and to break down echo chambers of self-reinforcing opinion. Using Twitter (now X) data, here we show the consequences of individual misinformation tagging: tagged posters had explored novel political information and expanded topical interests immediately prior, but being tagged caused posters to retreat into information bubbles. These unintended consequences were softened by a collective verification system for misinformation moderation. In Twitter’s new feature, Community Notes, misinformation tagging was peer-reviewed by other fact-checkers before revelat

**Minimum Cases**

In [74]:
bottom_cases = diff_df.nsmallest(3, 'difference')
for idx, case in bottom_cases.iterrows():
    print(f"\n{case['manipulation']} Manipulation:")
    print(f"  Title: {case['title'][:60]}...")
    print(f"  Original PPL: {case['original_ppl']:.2f}")
    print(f"  Manipulated PPL: {case['manipulated_ppl']:.2f}")
    print(f"  Difference: +{case['difference']:.2f} ({case['pct_change']:+.1f}%)")

    # Get the actual text
    paper_results = results_df[results_df['paper_id'] == case['paper_id']]
    original_text = paper_results[paper_results['manipulation'] == 'Original']['original_text'].values[0]
    manip_text = paper_results[paper_results['manipulation'] == case['manipulation']]['manipulated_text'].values[0]

    print(f"\n  ORIGINAL TEXT:")
    print(f"  {original_text}...")
    print(f"\n  MANIPULATED TEXT:")
    print(f"  {manip_text}...")


Complexity Manipulation:
  Title: Differential impact from individual versus collective misinf...
  Original PPL: 96.00
  Manipulated PPL: 91.52
  Difference: +-4.49 (-4.7%)

  ORIGINAL TEXT:
  Fears about the destabilizing impact of misinformation online have motivated individuals and platforms to respond. Individuals have increasingly challenged others’ online claims with fact-checks in pursuit of a healthier information ecosystem and to break down echo chambers of self-reinforcing opinion. Using Twitter (now X) data, here we show the consequences of individual misinformation tagging: tagged posters had explored novel political information and expanded topical interests immediately prior, but being tagged caused posters to retreat into information bubbles. These unintended consequences were softened by a collective verification system for misinformation moderation. In Twitter’s new feature, Community Notes, misinformation tagging was peer-reviewed by other fact-checkers before revel

**Investigating Jargon vs. Complexity Identical Stats**

In [75]:
jargon_diff = diff_df[diff_df['manipulation'] == 'Jargon'][['paper_id', 'difference']].rename(columns={'difference': 'jargon_diff'})
complexity_diff = diff_df[diff_df['manipulation'] == 'Complexity'][['paper_id', 'difference']].rename(columns={'difference': 'complexity_diff'})
comparison = jargon_diff.merge(complexity_diff, on='paper_id')
comparison['are_identical'] = np.isclose(comparison['jargon_diff'], comparison['complexity_diff'], atol=0.01)

print(f"\nNumber of papers where Jargon and Complexity have identical effects: {comparison['are_identical'].sum()}")
print(f"Total papers: {len(comparison)}")
print(f"Correlation between Jargon and Complexity effects: {comparison['jargon_diff'].corr(comparison['complexity_diff']):.3f}")

if comparison['are_identical'].sum() > 0:
    print("\nPapers with identical effects:")
    identical_papers = comparison[comparison['are_identical']]
    for _, row in identical_papers.iterrows():
        title = diff_df[diff_df['paper_id'] == row['paper_id']]['title'].values[0]
        print(f"  - {title[:60]}... (diff: {row['jargon_diff']:.2f})")


Number of papers where Jargon and Complexity have identical effects: 0
Total papers: 20
Correlation between Jargon and Complexity effects: 0.370


**High vs. Low Variability**

In [76]:
# Calculate per-paper average difference across all manipulations
paper_avg_diff = diff_df.groupby('paper_id')['difference'].mean().reset_index()
paper_avg_diff.columns = ['paper_id', 'avg_difference']
paper_avg_diff = paper_avg_diff.merge(diff_df[['paper_id', 'title']].drop_duplicates(), on='paper_id')

print("\nTop 3 Most Vulnerable Abstracts (highest avg increase):")
for _, row in paper_avg_diff.nlargest(3, 'avg_difference').iterrows():
    print(f"  - {row['title'][:60]}... (avg +{row['avg_difference']:.2f})")

print("\nTop 3 Most Resistant Abstracts (lowest avg increase):")
for _, row in paper_avg_diff.nsmallest(3, 'avg_difference').iterrows():
    print(f"  - {row['title'][:60]}... (avg +{row['avg_difference']:.2f})")



Top 3 Most Vulnerable Abstracts (highest avg increase):
  - Deep Neural Network Approaches for High-Dimensional Stochast... (avg +31.76)
  - Values in Crisis: How the Great Recession Reshaped the gover... (avg +22.41)
  - Differential impact from individual versus collective misinf... (avg +20.03)

Top 3 Most Resistant Abstracts (lowest avg increase):
  - A System-Agnostic Approach to Computational Ideology Detecti... (avg +3.43)
  - Diagnostic images for mild cognitive impairment reveal bioma... (avg +8.36)
  - Ethical Sourcing and Consumer Demand for Coffee... (avg +8.48)
